# Etching Rate（蝕刻速率）

和軟體用的是同一份計算（`magic/etching/model.py`）。**平常建議用軟體的「＋ 新增紀錄」精靈**：有示意圖、動畫、即時預覽，按確認才寫入，重複會提醒。

這裡適合：一次補登多筆、或自己做其他比較圖。紀錄 CSV 的位置在軟體的 ⚙ 設定 → 蝕刻紀錄。

**兩種量法**（可以混用）

| | 公式 | 需要填 |
|---|---|---|
| SiO2：marker | d_SiO2 = H_m,out − T'_m（沒量 T'_m 就用 T_m，假設 Cr/Au 不會被蝕刻） | `marker_before`、`marker_out`（`marker_after_own` 可選） |
| SiO2：AFM 台階 | d_SiO2 = 台階 | `sio2_step` |
| SiO2：橢偏儀 | d_SiO2 = h − h' | `sio2_before`、`sio2_after` |
| 材料：從基板量 | d = T + d_SiO2 − H_out | `mat_before`、`mat_out` |
| 材料：圖案內台階 | d = S | `mat_step`（`mat_before` 可選，用來判斷蝕穿） |

- 每個量測欄位都可以填好幾個點（`"80.1, 79.9, 80.3"`）→ 自動取平均 ± 標準差、誤差傳遞
- 時間可以分段：`"20+20+20"`、`"3x20"`、`"1.5min"`
- 剩下 ≤ max(0.5 nm, 2σ) 判定為蝕穿 → 只能得到下限 r ≥ T / t


In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from magic.core import settings as S
from magic.core.plotstyle import apply
from magic.etching.model import EtchInput, compute
from magic.etching import store
from magic.runner import run
apply()
CSV = S.etching_csv()          # 和軟體同一份紀錄
print("紀錄檔：", CSV, "（格式：", store.file_schema(CSV), "）")

## 計算（先看結果，確認後再寫入）

In [ ]:
inp = EtchInput(
    material="hBN", resist="PMMA", line_width="2um", name="",
    instrument="ICP-RIE", gases=[("SF6", "40")], icp_power="100", rf_power="30", pressure="10", pressure_unit="mTorr",
    steps="3x20",
    sio2_method="marker", marker_before="50.2, 50.0", marker_out="53.1, 52.9",
    mat_method="outside", mat_before="80.1, 79.9, 80.3", mat_out="45.2, 45.8, 45.0",
)
res = compute(inp)
print("\n".join(res.explain))
for w in res.warnings:
    print("⚠", w)

In [ ]:
WRITE = False      # ← 確認上面的數字沒問題再改成 True
from magic import __version__
rec = store.make_record(inp, res, version=__version__)
dup = store.find_duplicates(store.read_log(CSV), rec)
if dup:
    print("和這幾筆重複：", dup)
if WRITE and not dup:
    if store.file_schema(CSV) == "v1":
        print("舊格式，先備份再升級：", store.upgrade_file(CSV))
    store.append_record(rec, CSV)
    print("已寫入")

## 比較（篩選取代原本寫死的 row_indices）

In [ ]:
df = store.read_log(CSV)
if df.empty:
    from magic.testdata import etching_csv_v2
    demo = os.path.join(str(S.data_dir()), "demo", "Etching", "etch_rate_log.csv")
    etching_csv_v2(demo)
    df = store.read_log(demo)
    print("紀錄是空的，先用範例資料（合成）")
sel = df[df["Method"] == "ICP-RIE"]             # ← 例：只看 ICP-RIE
fig, res = run("Etching", "條件比較（平均 ± 標準差）", [sel], by="Line width (nm)", by2="Gas")
print(res.text)
fig

## 多個時間線性擬合：斜率 = 蝕刻率，截距 → 起始延遲

In [ ]:
fig, res = run("Etching", "多時間線性擬合", [df], by_width=True, weighted=True)
res.tables["time_fit"]

## 不同線寬的蝕刻率（其他條件相同才會放在同一條線）

In [ ]:
fig, res = run("Etching", "線寬比較", [df], log_x=True)
print(res.text)
fig